In [1]:
from data.real.DataPreparation import DataPreparation
from evaluation.evalutation import eval
from utils import *
import torch
import os
import yaml
import pandas as pd
import random
import numpy as np
import sys
import random
from torch import nn, optim
from typing import List, Tuple
import plotly.graph_objects as go

import warnings

warnings.filterwarnings("ignore")

/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = {}
if os.path.exists("config.yaml"):
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

experiments = config.get("experiments", [])
n_repeat_exps = 5
if not experiments:
    print("No experiments found in config.yaml under 'experiments'. Exiting.")
    sys.exit(1)

def set_seed(seed: int = 42):
    print(f"Seed: {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

In [3]:
def local_train_frogdq_new(
    X: torch.Tensor,
    y: torch.Tensor,
    model,
    g_global: torch.Tensor,
    q: torch.Tensor,
    mu: float = 0.1,           # proximal-to-global strength (same as before)
    lr: float = 0.01,          # step size used for both w and g
    epochs: int = 1,
    freeze: bool = False,
) -> tuple:
    inv_q = (1.0 - q)
    if isinstance(model, torch.Tensor):
        # Logistic regression with intercept (w[0] is intercept, w[1:] are feature weights)
        w = model.clone().detach().requires_grad_(True)
        g = g_global.clone().detach().requires_grad_(True)
        if freeze:
            g_global.clone().detach().requires_grad_(False)
        #opt = optim.SGD([w, g], lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()
        for i in range(epochs):
            # Forward pass
            logits = (X * g) @ w[1:] + w[0]   # g is gating mask
            data_loss = loss_fn(logits, y)
            prox = 0.5 * mu * torch.sum(inv_q * (g - g_global) ** 2)
            loss = data_loss + prox

            # Backward pass
            loss.backward()

            # Manual SGD update
            with torch.no_grad():
                w -= lr * w.grad
                if not freeze:
                    g -= lr * g.grad

            # Zero gradients manually
            w.grad.zero_()
            if not freeze:
                g.grad.zero_()
        return w.detach(), g.detach()

In [4]:
def run_federated_training(
    X_clients: List[torch.Tensor],
    y_clients: List[torch.Tensor],
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    w_global: torch.Tensor,
    n_rounds: int = 100,
    local_epochs: int = 1,
    lr: float = 0.01,
    mu: float = 0.1,
    limit_round: int = 100,
    q_clients: List[torch.Tensor] = None,
    no_temp: bool = False
) -> Tuple[object, object]:
   
    print(f"Limit Round: {limit_round}")
    d = X_clients[0].shape[1]
    n_clients = len(X_clients)
    n_samples = torch.tensor([len(x) for x in X_clients], dtype=torch.float32)
    #w_global = torch.randn(d+1) / torch.sqrt(torch.tensor(d + 1.0)) #init values
    #w_global = torch.zeros(d+1)
    g_global = torch.ones(d) #init values
    w_global_lst = [w_global]
    g_global_lst = [g_global]
    accuracy_val_round = []
    loss_val_round = []
    roc_auc_val_round = []

    for rnd in range(n_rounds):
        w_updates, g_updates = [], []
        for k in range(n_clients):
            w_k, g_k = local_train_frogdq_new(
                X=X_clients[k],
                y=y_clients[k],
                model=w_global,
                g_global=g_global,
                q=q_clients[k],
                mu=mu*((1.015)**rnd) if (rnd > 0 and rnd < limit_round) else mu,
                lr=lr,
                epochs=local_epochs,
                freeze=True if (rnd > limit_round and no_temp == False) else False
            )
            w_updates.append(w_k)
            g_updates.append(g_k)

        stacked_w = torch.stack(w_updates)
        w_global = (stacked_w.T @ n_samples / n_samples.sum()).T
        # Save w global aggregated values after the end of each round
        w_global_lst.append(w_global)

        if rnd <= limit_round or no_temp == True:
            stacked_g = torch.stack(g_updates)
            g_global = (stacked_g.T @ n_samples / n_samples.sum()).T
            # Save g global aggregated values after the end of each round
            g_global_lst.append(g_global)

        # Compute Accuracy on Validation test after aggregation
        roc_auc_val, accuracy_val, loss_val = eval(X=X_val, y=y_val, w=w_global, g=g_global)
        accuracy_val_round.append(accuracy_val)
        loss_val_round.append(loss_val)
        roc_auc_val_round.append(roc_auc_val)

    return w_global, g_global, accuracy_val_round, roc_auc_val_round, loss_val_round

In [5]:
for exp in experiments:
    print(f"\n===== Running Experiment: {exp.get('name', 'Unnamed')} =====")
    if exp.get('name', 'Unnamed') == 'Experiment Real':
        # Set seed for reproducibility (optional: allow per-experiment seed)
        # set_seed(exp.get('seed', 42))

        # Extract parameters for this experiment
        experiment_name = exp.get("name", "Unnamed")
        K = exp.get("K", 10)
        N = exp.get("N", 1000)
        D = exp.get("D", 20)
        n_rounds = exp.get("n_rounds", 10)
        local_epochs = exp.get("local_epochs", 1)
        lr = exp.get("lr", 0.2)
        mu = exp.get("mu", 1.0)
        n_corrupt = exp.get("n_corrupt", 5)
        synthetic_data = exp.get("synthetic_data", False)
        dataset_path = exp.get("dataset_path", "")
        target_column = exp.get("target_column", "")
        model_type = exp.get("model_type", "logreg")
        num_classes = exp.get("num_classes", 2)
        n_feature_show = exp.get("n_feature_show", 5)
        noise_std_syn = exp.get("noise_std_syn", 0.2)
        noise_std_poisoning = exp.get("noise_std_poisoning", 0.2)
        missing = exp.get("missing", False)

        # Set fixed seeds for reproducibility per experiment
        seeds = [43, 56, 221, 800, 1258]
        if len(seeds) < n_repeat_exps:
            raise ValueError(
                f"Not enough seeds for {n_repeat_exps} repetitions. Please provide at least {n_repeat_exps} seeds."
            )

        exp_type = {
            "100% Rounds": {
                "percentage": 1.0,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
            "70% Rounds": {
                "percentage": 0.7,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
            "40% Rounds": {
                "percentage": 0.4,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
            "10% Rounds": {
                "percentage": 0.1,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
            "1 Round": {
                "percentage": 0.002,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
            "No Temp": {
                "percentage": 0.0,
                "acc_val": [],
                "roc_auc_val": [],
                "loss_val": []
            },
        }

        for i in range(n_repeat_exps):
            set_seed(seeds[i])
            corr_features = ['hours_worked_per_week', 'education', 'occupation', 'final_weight', 'is_male']
            data_prep = DataPreparation(dataset_name='adult')
            data_prep.load(label_col='over_threshold')
            data_prep.set_feature_poisoning(corr_features)
            X_clients, X_corr_clients, y_clients, X_test, y_test, X_val, y_val, q_clients = data_prep.split_across_client(number_clients=K, label_col='over_threshold')
            print(f"q_clients: {q_clients}")

            X_clients_no_corr = []
            for X in X_clients:
                X_clients_no_corr.append(X[:, q_clients[0] == 1])
            
            X_val_no_corr = X_val[:, q_clients[0] == 1]
            X_test_no_corr = X_test[:, q_clients[0] == 1]
            

            # ---------------- Run Experiments ----------------
            w_global = torch.randn(X_clients[0].size(1)+1) / torch.sqrt(torch.tensor(X_clients[0].size(1)+1))
            for single_exp in exp_type:
                w, g, acc_val, roc_auc_val, loss_val = run_federated_training(
                    X_clients=X_corr_clients,  
                    y_clients=y_clients,
                    X_val=X_val,
                    y_val=y_val,
                    n_rounds=n_rounds,
                    local_epochs=local_epochs,
                    lr=lr,
                    mu=mu,
                    q_clients=q_clients,
                    w_global=w_global,
                    limit_round=int(n_rounds*exp_type[single_exp]["percentage"]),
                    no_temp=True if single_exp == 'No Temp' else False
                )
                roc_auc_test, accuracy_test, loss_test = eval(
                    X=X_test, y=y_test, w=w, g=g
                )
                print(
                    f"{exp_type[single_exp]} accuracy: {accuracy_test:.3f} / roc_auc: {roc_auc_test:.3f}"
                )

                exp_type[single_exp]["acc_val"].append(acc_val)
                exp_type[single_exp]["roc_auc_val"].append(roc_auc_val)
                exp_type[single_exp]["loss_val"].append(loss_val)


===== Running Experiment: Experiment 1 =====

===== Running Experiment: Experiment 2 =====

===== Running Experiment: Experiment 3 =====

===== Running Experiment: Experiment 4 =====

===== Running Experiment: Experiment Real =====
Seed: 43
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_train_corr: torch.Size([1752, 91])
X_train: torch.Size([1752, 91]), y_train: torch.Size([1752]), X_t

In [6]:
fig_acc = go.Figure()
fig_loss = go.Figure()
fig_auc_roc = go.Figure()

for i, sim_name in enumerate(exp_type):
    if sim_name:
        fig_acc.add_trace(
            go.Scatter(
                y=np.mean(exp_type[sim_name]["acc_val"], axis=0),
                mode="lines",
                name=f"{sim_name}",
            )
        )

        fig_loss.add_trace(
            go.Scatter(
                y=np.mean(exp_type[sim_name]["loss_val"], axis=0),
                mode="lines",
                name=f"{sim_name}",
            )
        )

        fig_auc_roc.add_trace(
            go.Scatter(
                y=np.mean(exp_type[sim_name]["roc_auc_val"], axis=0),
                mode="lines",
                name=f"{sim_name}",
            )
        )

fig_acc.update_layout(
    title="Accuracy on Validation Set over rounds",
    xaxis_title="Rounds",
    yaxis_title="Accuracy",
    template="simple_white",
)

fig_loss.update_layout(
    title="Loss on Validation Set over rounds",
    xaxis_title="Rounds",
    yaxis_title="Loss",
    template="simple_white",
)

fig_auc_roc.update_layout(
    title="ROC-AUC on Validation Set over rounds",
    xaxis_title="Rounds",
    yaxis_title="ROC-AUC",
    template="simple_white",
)

os.makedirs(f'data_visualization/graphics/Temperature Frog/', exist_ok=True)

fig_acc.write_image(
    f"data_visualization/graphics/Temperature Frog/accuracy_rounds.png", width=800, height=600, scale=1
)
fig_loss.write_image(
    f"data_visualization/graphics/Temperature Frog/loss_rounds.png", width=800, height=600, scale=1
)
fig_auc_roc.write_image(
    f"data_visualization/graphics/Temperature Frog/roc_auc_rounds.png", width=800, height=600, scale=1
)